In [ ]:
import numpy as np 
import pandas as pd 
import os
import sys
import requests
import shutil
import torch
import torchvision.transforms as transforms
import torchvision.datasets as datasets
from torch.utils.data import DataLoader
import timm
import kagglehub


print("Installing dependencies and cloning repositories...")
!pip install -q timm yacs iopath kagglehub
!pip install -q git+https://github.com/fra31/auto-attack.git

os.chdir('/content')
if os.path.exists('/content/SAR'): shutil.rmtree('/content/SAR')
if os.path.exists('/content/test-time-adaptation'): shutil.rmtree('/content/test-time-adaptation')

!git clone https://github.com/mr-eggplant/SAR.git /content/SAR
!git clone --depth 1 https://github.com/mariodoebler/test-time-adaptation.git /content/test-time-adaptation


sys.path.append('/content/SAR')
sys.path.insert(0, '/content/test-time-adaptation/classification')


import sar
def stable_softmax_entropy(x):
    return -(x.softmax(1) * x.log_softmax(1)).sum(1)
sar.softmax_entropy = stable_softmax_entropy


import types
import robustbench
try:
    import robustbench.loaders as rb_loaders
except ImportError:
    rb_loaders = types.ModuleType('loaders')
    robustbench.loaders = rb_loaders
    sys.modules['robustbench.loaders'] = rb_loaders

class DummyDataset(torch.utils.data.Dataset):
    def __init__(self, *args, **kwargs): pass
    def __len__(self): return 0
rb_loaders.CustomCifarDataset = DummyDataset
rb_loaders.CustomImageFolder = datasets.ImageFolder


In [ ]:
data_path = '/kaggle/input/datasets/sampadramkumar/gaussian-noise/kaggle/working/gaussian_noise_5'

# The Attack

In [ ]:
import copy
import torch.nn as nn
import torch.nn.functional as F
from torchvision.models import resnet50, ResNet50_Weights
from tqdm import tqdm
import timm

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")


print("Initializing TSMA Surrogate (Standard ResNet-50)...")
attack_model_resnet = timm.create_model('resnet50', pretrained=True).to(device).eval()
for param in attack_model_resnet.parameters():
    param.requires_grad = False


def tsma_attack_optimized(tta_model, images, labels, epsilon=8/255, alpha=2/255, steps=5, decay=1.0, lambda_drift=0.2):

    mean_std = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1).to(images.device)
    std_std = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1).to(images.device)
    images_unnorm = (images * std_std) + mean_std

    
    with torch.no_grad():
        clean_features = attack_model_resnet.forward_features(images)
        clean_mu = clean_features.mean(dim=(2, 3))
        clean_var = clean_features.var(dim=(2, 3))

    delta = torch.zeros_like(images_unnorm).uniform_(-epsilon, epsilon)
    delta = torch.clamp(images_unnorm + delta, min=0, max=1) - images_unnorm
    delta.requires_grad_(True)
    momentum = torch.zeros_like(delta).detach()

    
    with torch.enable_grad():
        for _ in range(steps):
            adv_images_unnorm = images_unnorm + delta
            adv_images = (adv_images_unnorm - mean_std) / std_std

            outputs = attack_model_resnet(adv_images)
            adv_features = attack_model_resnet.forward_features(adv_images)

            adv_mu = adv_features.mean(dim=(2, 3))
            adv_var = adv_features.var(dim=(2, 3))

            
            loss_ce_true = F.cross_entropy(outputs, labels)

            
            probs = F.softmax(outputs, dim=1)
            loss_entropy = -torch.sum(probs * torch.log(probs + 1e-6), dim=1).mean()

            
            loss_drift = F.mse_loss(adv_mu, clean_mu) + F.mse_loss(adv_var, clean_var)

            loss = -loss_ce_true + (1.5 * loss_entropy) - (lambda_drift * loss_drift)

            if delta.grad is not None:
                delta.grad.zero_()

            loss.backward()

            
            with torch.no_grad():
                grad = delta.grad
                grad_norm = torch.norm(grad.view(grad.shape[0], -1), p=1, dim=1).view(-1, 1, 1, 1)
                grad_norm = torch.clamp(grad_norm, min=1e-8)
                grad_normalized = grad / grad_norm

                momentum = decay * momentum + grad_normalized

                delta.data = delta.data - alpha * momentum.sign()
                delta.data = torch.clamp(delta.data, min=-epsilon, max=epsilon)
                delta.data = torch.clamp(images_unnorm + delta.data, min=0, max=1) - images_unnorm

    final_adv_images_unnorm = images_unnorm + delta.detach()
    return (final_adv_images_unnorm - mean_std) / std_std



class ROID(nn.Module):
    def __init__(self, model, lr=2.5e-4, alpha=0.99, beta=0.9, tau=1/3):
        super().__init__()
        self.model = model
        self.model_0 = copy.deepcopy(model)
        self.model_0.requires_grad_(False)
        self.model_0.eval()

        params_to_train = []
        for m in self.model.modules():
            if isinstance(m, nn.BatchNorm2d):
                m.requires_grad_(True)
                m.track_running_stats = False
                m.running_mean = None
                m.running_var = None
                params_to_train.append(m.weight)
                params_to_train.append(m.bias)
            else:
                m.requires_grad_(False)

        self.optimizer = torch.optim.SGD(params_to_train, lr=lr, momentum=0.9)
        self.alpha = alpha
        self.beta = beta
        self.tau = tau
        self.tendency = None

    def slr_loss(self, probs):
        probs = torch.clamp(probs, min=1e-5, max=1.0 - 1e-5)
        neg_probs = 1.0 - probs
        slr = probs * torch.log(probs / neg_probs)
        return -slr.sum(dim=1)

    def sce_loss(self, p1, p2):
        p1 = torch.clamp(p1, min=1e-5, max=1.0 - 1e-5)
        p2 = torch.clamp(p2, min=1e-5, max=1.0 - 1e-5)
        loss1 = -(p1 * torch.log(p2)).sum(dim=1)
        loss2 = -(p2 * torch.log(p1)).sum(dim=1)
        return 0.5 * (loss1 + loss2)

    def adapt(self, x):
        self.model.train()
        x_aug = transforms.functional.hflip(x)

        outputs = self.model(x)
        outputs_aug = self.model(x_aug)

        probs = outputs.softmax(dim=1)
        probs_aug = outputs_aug.softmax(dim=1)

        batch_mean_probs = probs.mean(dim=0).detach()
        if self.tendency is None:
            self.tendency = batch_mean_probs
        else:
            self.tendency = self.beta * self.tendency + (1 - self.beta) * batch_mean_probs

        sim = F.cosine_similarity(probs.detach(), self.tendency.unsqueeze(0), dim=1)
        w_div = 1.0 - sim

        ent = -(probs.detach() * torch.log(probs.detach() + 1e-6)).sum(dim=1)
        w_cert = -ent

        def normalize_01(w):
            w_min, w_max = w.min(), w.max()
            if w_max > w_min: return (w - w_min) / (w_max - w_min)
            return w

        w_final = torch.exp((normalize_01(w_div) * normalize_01(w_cert)) / self.tau)
        w_final = w_final / (w_final.mean() + 1e-8)

        loss_slr = self.slr_loss(probs)
        loss_cons = self.sce_loss(probs, probs_aug)

        loss = (w_final * loss_slr).mean() + loss_cons.mean()

        if loss.requires_grad:
            self.optimizer.zero_grad()
            loss.backward()
            self.optimizer.step()

        with torch.no_grad():
            for p, p0 in zip(self.model.parameters(), self.model_0.parameters()):
                if p.requires_grad:
                    p.data = self.alpha * p.data + (1 - self.alpha) * p0.data

        self.model.eval()

        with torch.no_grad():
            final_outputs = self.model(x)
            final_probs = final_outputs.softmax(dim=1)

            p_hat = final_probs.mean(dim=0)
            N_b = final_probs.size(0)
            N_c = final_probs.size(1)

            gamma = max(1.0/N_b, 1.0/N_c) / (p_hat.max() + 1e-6)
            p_bar = (p_hat + gamma) / (1 + gamma * N_c)

            corrected_probs = final_probs * (p_bar * N_c).unsqueeze(0)
            corrected_probs = corrected_probs / corrected_probs.sum(dim=1, keepdim=True)

        return corrected_probs


print("Loading standard ImageNet ResNet-50 for ROID...")
model = resnet50(weights=ResNet50_Weights.IMAGENET1K_V1).to(device)

data_path = '/kaggle/input/datasets/sampadramkumar/gaussian-noise/kaggle/working/gaussian_noise_5'

timm_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])


print(f"Loading data from: {data_path}")
dataset = datasets.ImageFolder(root=data_path, transform=timm_transform)

print("Remapping WordNet IDs to ImageNet classes...")
url = "https://s3.amazonaws.com/deep-learning-models/image-models/imagenet_class_index.json"
mapping = requests.get(url).json()
wnid_to_idx = {v[0]: int(k) for k, v in mapping.items()}

new_samples = [(p, wnid_to_idx[os.path.basename(os.path.dirname(p))])
               for p, _ in dataset.samples if os.path.basename(os.path.dirname(p)) in wnid_to_idx]
dataset.samples = new_samples
dataset.targets = [s[1] for s in new_samples]

print(f"Dataset ready. Total images: {len(dataset)}")

dataloader = DataLoader(dataset, batch_size=64, shuffle=True, num_workers=4, pin_memory=True)


roid = ROID(model, lr=2.5e-4).to(device)

correct_total, total_total = 0, 0
correct_clean, total_clean = 0, 0
correct_pois, total_pois = 0, 0

POISON_FREQ = 2  

print("\nStarting Online Test-Time Adaptation against Poisoned TSMA Attack...")
pbar = tqdm(enumerate(dataloader), total=len(dataloader), desc="ROID ResNet-50 (Poisoned)")

for i, (images, labels) in pbar:
    images = images.to(device).float()
    labels = labels.to(device)

    is_poisoned = (i % POISON_FREQ == 0)
    
    if is_poisoned:
        inputs = tsma_attack_optimized(
            tta_model=None, 
            images=images, 
            labels=labels,
            epsilon=8/255, 
            alpha=2/255, 
            steps=5, 
            decay=1.0, 
            lambda_drift=0.2
        )
    else:
        inputs = images

    
    outputs = roid.adapt(inputs)
    
    preds = outputs.argmax(dim=1)

    batch_correct = (preds == labels).sum().item()
    batch_total = labels.size(0)
    batch_acc = 100.0 * batch_correct / batch_total

    
    correct_total += batch_correct
    total_total += batch_total

    if is_poisoned:
        correct_pois += batch_correct
        total_pois += batch_total
    else:
        correct_clean += batch_correct
        total_clean += batch_total

    
    total_acc = 100.0 * correct_total / max(1, total_total)
    clean_acc = 100.0 * correct_clean / max(1, total_clean)
    pois_acc = 100.0 * correct_pois / max(1, total_pois)

    
    pbar.set_postfix({
        "Total": f"{total_acc:.2f}%",
        "Batch": f"{batch_acc:.2f}%",
        "Clean": f"{clean_acc:.2f}%",
        "Pois": f"{pois_acc:.2f}%",
        "Pois?": "Y" if is_poisoned else "N"
    })

print("\nFINAL RESULTS")
print(f"Total Accuracy : {100.0 * correct_total / total_total:.2f}%")
print(f"Clean Accuracy : {100.0 * correct_clean / max(1, total_clean):.2f}%")
print(f"Poison Accuracy: {100.0 * correct_pois / max(1, total_pois):.2f}%")